# GPU runner (Kaggle or Colab)

Clones the repo, installs it, and runs one training command. Results end up in `results.zip`; unzip it into `reports/results/` locally and commit.

**One-time setup**
- Only if the repo is private: a read-only GitHub token stored as a secret named `GITHUB_TOKEN` (Kaggle: Add-ons → Secrets; Colab: the key icon). Public repo: nothing to do.
- **Kaggle:** Settings → Accelerator **GPU T4 x2 or P100**, Internet **on** (needs a phone-verified account), and *Add Input* → competition **asap-aes**.
- **Colab:** Runtime → change runtime type → **T4 GPU**. Put `training_set_rel3.tsv` (or its .zip) in Google Drive under `MyDrive/asap-aes/`.

For long runs on Kaggle, use **Save Version → Save & Run All** so the run continues with the tab closed.

This notebook never prints essay text. The competition rules forbid sharing it.

In [ ]:
# ---- What to run ----
BRANCH = "main"
CONFIG = "configs/deberta_v3_base.yaml"
FOLDS = "0 1 2 3 4"
EXTRA = ""            # e.g. "--seed 43" or "--limit 64 --epochs 1" for a quick test

In [ ]:
import glob, os, shutil, subprocess, zipfile

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # not os.path.exists("/kaggle"): Colab has that folder too
REPO = "Dilip0552/essay-grader-lstm-to-llm"
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
CODE = "/tmp/repo"  # outside WORK so the token-bearing clone is never saved as notebook output

token = ""  # only needed while the repo is private
try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    else:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")

os.chdir(WORK)  # leave CODE before deleting it, so re-running this cell works
shutil.rmtree(CODE, ignore_errors=True)
r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, f"https://{token + '@' if token else ''}github.com/{REPO}.git", CODE],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError(r.stderr.replace(token, "***") if token else r.stderr)
subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements.txt -e .

In [ ]:
# Find the ASAP data (Kaggle input or Google Drive), unzip if needed
roots = ["/kaggle/input"] if ON_KAGGLE else ["/content/drive/MyDrive/asap-aes"]
hits = [p for r in roots for p in glob.glob(f"{r}/**/training_set_rel3.tsv*", recursive=True)]
assert hits, f"training_set_rel3.tsv not found under {roots}"
tsv = next((p for p in hits if p.endswith(".tsv")), None)
if tsv is None:
    zipfile.ZipFile(hits[0]).extractall("/tmp/asap")
    tsv = "/tmp/asap/training_set_rel3.tsv"
os.environ["ASAP_TSV"] = tsv
!python scripts/download_data.py

In [ ]:
# Runs folds in parallel when there are 2 GPUs (Kaggle T4 x2), sequentially otherwise
import torch
n_gpu = max(torch.cuda.device_count(), 1)
folds = FOLDS.split()
procs = []
for g in range(n_gpu):
    mine = folds[g::n_gpu]
    if mine:
        cmd = f"python scripts/train_transformer.py --config {CONFIG} --folds {' '.join(mine)} {EXTRA}"
        log = open(f"{WORK}/train_gpu{g}.log", "w")
        procs.append(subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT,
                                      env={**os.environ, "CUDA_VISIBLE_DEVICES": str(g)}))
        print(f"GPU {g}: folds {mine}  (log: {WORK}/train_gpu{g}.log)")
codes = [p.wait() for p in procs]
for g in range(len(procs)):
    print(f"--- GPU {g} results ---")
    print("".join(l for l in open(f"{WORK}/train_gpu{g}.log") if l.startswith(("{", "[", "Traceback", "  File", "RuntimeError", "ValueError"))))
assert not any(codes), f"exit codes {codes}: check the logs above"


In [ ]:
shutil.make_archive(f"{WORK}/results", "zip", "reports/results")
print("saved", f"{WORK}/results.zip")
if not ON_KAGGLE:
    from google.colab import files
    files.download(f"{WORK}/results.zip")